# Prediction: Forecasting Cases You Have Not Seen

**Topic 07 · 1 lecture**

<hr>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb07_prediction_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** prediction (descriptive kind · unseen-cases reach). A
**prediction** is a best guess about a case whose outcome you cannot see yet.
Example: a weather app guessing whether it rains tomorrow, before tomorrow
exists. Prediction is descriptive, not causal: it forecasts *who* or *what*,
never *why*. This week you learn to forecast honestly, which means beating a
dumb baseline on cases the model never studied, and knowing when a good-looking
score is fake.

**Design pathway:** prediction. This is the one design-library pathway your
course book covers only in passing, so the course writes the missing entry,
**"Observational: predictive,"** in the book's own declare-diagnose-redesign
grammar. You watch and record without intervening, so you can forecast and,
with care, generalize to new cases, but you cannot read a cause off a model's
weights.

| | |
|---|---|
| **A claim this topic PERMITS** | "On held-out cases the model never trained on, my model beats the baseline by [margin] on [metric]; I checked for leakage, and I make no causal reading of its weights." |
| **A claim this topic does NOT permit** | Any performance number computed on data the model already saw, or "the top feature is X, so X drives the outcome." Importance is not explanation, and leakage can fake importance. |

**Where this sits in the course:** Week 8 of the design library. It follows the
observational, experimental, and measurement pathways, and it develops M7, your
declared analysis protocol, which faces a prediction-and-leakage audit at this
week's Friday studio. Next week the causal question opens.

*Provenance: course-authored 'Observational: predictive' library entry (declare-diagnose-redesign format, modeled on the DeclareDesign design library at book.declaredesign.org/library) + scikit-learn + rdss la_voter_file | prediction vs description/explanation/causation, baselines, train/validation/test, cross-validation, metrics, calibration, overfitting, leakage, distribution shift, subgroup performance, model-selection and not-useful rules | seeded baseline-vs-model on held-out voter-file rows plus a leaky feature you build (seed 464) | fresh (course-authored library entry)*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Separate **prediction** from description, explanation, and causation, and say
   which one a forecasting question really asks.
2. State the prediction **contract** in order (target, baseline, split, metric)
   and score an honest baseline on **held-out** cases before trusting any model.
3. Show, with seeded code, that a real model beats a dumb baseline by only a
   modest margin, and read that modest win as a success, not a disappointment.
4. Build a **data leak** yourself, watch it fake a great score, then catch it by
   correlation and timing and retrain clean.
5. Use **cross-validation**, a **calibration** check, a **subgroup** check, and a
   **distribution-shift** check to tell a real win from a lucky or fragile one,
   and name the criteria that make a model **not useful**.
6. Apply the target-baseline-split-metric-leakage spine to your own project's M7
   declared analysis protocol, or defend in writing why prediction is the wrong
   question for it.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx  # for the leakage-timing diagram; if missing locally: pip install networkx (pre-installed in Colab)

# scikit-learn: the prediction workflow (split, baseline, model, metrics, calibration).
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.calibration import calibration_curve

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

### 🤝 AI Research Partner

This week you work with **Gemini** as a research partner on a task where AI is
genuinely useful and genuinely dangerous. Useful, because it can explain a
scikit-learn cell line by line, list candidate leaks in your feature set, or
attack a headline you wrote. Dangerous, because a fluent walkthrough can describe
numbers your code never printed, and a confident metric recommendation can be
exactly wrong for your target.

**Never delegate these:** the target you are predicting, the baseline you must
beat, whether a feature would exist at prediction time, and the verdict on
whether your project should predict at all. Those are yours to declare and
defend. The full list of never-delegate decisions lives in
[`ai_resources/human_responsibility_checklist.md`](https://github.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/ai_resources/human_responsibility_checklist.md).

**Commit first, then ask.** Every AI block below asks you to write your own
answer before you open Gemini, and to check the tool's claims against the numbers
your cell actually printed. Code that runs without error is not the same as code
that computed the right thing.

> **A question that often comes up here:** *"Gemini writes model code faster than
> I can. Why not let it drive the whole analysis?"* Because the hard part of
> prediction is not the code. It is deciding what counts as a fair test, and a
> tool that has never seen your data's timeline cannot know which feature secretly
> holds the answer. Writing the contract yourself keeps you the researcher and
> keeps Gemini the assistant.

---

# Lecture 1

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

A classmate proudly shows a model that predicts, with **99% accuracy**, whether
each registered voter **already voted** in a past election. The code runs clean,
the accuracy is real arithmetic, and the number is genuinely 99%.

Here is the claim on the table: **"This model is a breakthrough forecaster of
turnout."** Would you stake your name on it, yes or no? Commit to one answer
before we go further. Then defend it. A model that predicts a *past* event from
the voter file has one suspicious advantage a real forecaster never gets. Name
what that advantage might be: which single column in the inputs could quietly
already contain the answer? If such a column exists, the 99% is not skill. It is
the model reading the answer key. Finding that column by reasoning, before any
code, is the whole job of this lecture.

## 1. What Kind of Question Is a Forecast?

**Guiding question:** *can you forecast new cases better than the dumbest honest
rule, and would you even notice if your score were secretly leaking?*

> *"Everyone brings me a model that scores ninety-something percent. I have one
> question before I fund anything. Ninety-something compared to WHAT? In my
> world, guessing 'no' every single time already scores 88. If your model cannot
> beat that, you have not built anything. You have dressed up the base rate in a
> lab coat."*
> — a **policy stakeholder** who has been burned by impressive-sounding accuracy

Prediction is one of the four compass positions, and it is the one most likely to
fool you. To place it, separate four questions that beginners blur together:

- **Description** asks what is true in the data you have. Example: what share of
  these voters turned out. It stays inside the cases at hand.
- **Explanation** asks how the parts of a system relate. Example: which voter
  traits tend to go with turnout, as a pattern.
- **Causation** asks what would change turnout if you intervened. Example: does a
  mailer *make* someone vote. That needs a counterfactual and a design.
- **Prediction** asks for a best guess on a case you have not seen. Example: for a
  newly registered voter, will they turn out next time. It reaches to unseen
  cases, and it is graded only by whether the guess is right.

Prediction's reach is **unseen cases**: units that are not in your data yet,
whose outcome is still unknown. Example: next year's registrants. That reach is
what makes prediction its own position, and it is why a prediction is only ever
tested one honest way, on cases the model did not get to study first.

> **A question that often comes up here:** *"Isn't a model with a high accuracy
> automatically a good model?"* No, and this is the trap the stakeholder is
> warning about. Accuracy means nothing on its own. It only means something next
> to two other things: the **baseline** you had to beat, and proof the score came
> from cases the model never saw. A 99% that read the answer key is worth less
> than a 64% that was earned on genuinely new cases.

A forecast earns trust only under a four-part **contract**, and the order is not
negotiable. Learn each term now, because your M7 declared analysis protocol is
graded on stating all four before you touch your data.

- **Target**: the one thing you are predicting, a single column named before
  anything else. Example: `voted_2012`, whether each person voted.
- **Baseline**: the dumbest honest rule you must beat. Usually "always guess the
  most common answer." Example: if 60% voted, always guessing "voted" scores 60%
  for free, so 60% is the bar.
- **Split**: dividing your rows into a **training set** the model learns from and
  a **held-out set** it never touches during training. A **held-out set** is a
  batch of rows you lock in a drawer while the model studies, then use as its
  exam. Example: keep a random 25% of voters aside, train on the other 75%.
- **Metric**: how you keep score, chosen to match the target. Example: accuracy,
  the share of guesses that were right, works when classes are roughly balanced;
  it lies when one outcome is rare.

Skip a step and the failure is predictable. No baseline, and you cannot tell
skill from the base rate. No held-out split, and you are grading the model on its
own homework. Wrong metric, and you optimize the wrong thing beautifully. Load
the voter file and look before you model anything.

In [ ]:
# Step 0 of any prediction task: LOOK at the data before modeling it.
voters = load_course_data("la_voter_file.csv")
assert voters.shape == (1000, 4), "unexpected shape — flag this!"
print("✓ Loaded la_voter_file.csv —", voters.shape[0], "rows,", voters.shape[1], "columns")
print("  Columns:", list(voters.columns))
print()

base_rate = voters["voted_2012"].mean()
print("TARGET = voted_2012 (did this person vote in 2012?)")
print(f"  base rate (share who voted): {base_rate:.1%}")
print(f"  class balance: {voters['voted_2012'].value_counts().to_dict()}")
print(f"  'party' is text with {voters['party'].nunique()} categories; "
      f"'census_tract' is an ID code with {voters['census_tract'].nunique()} distinct values.")

Three things in that printout decide everything below. **First**, the majority
answer is "voted" (603 of 1,000), so the rule "always say voted" scores about
**60.3% by construction**. That 60.3%, the share of the most common outcome, is the
**base rate**, and it is your baseline: any model has to beat it to have earned
anything. **Second**, `party` is text, so the code below one-hot
encodes it (turns each category into its own 0/1 column) before a model can read
it. **Third**, `census_tract` is an ID code wearing a number's clothes. Tract
408141 is not "bigger" than tract 190520; the digits are just a label. Feeding
hundreds of ID codes to a model is a fast way to overfit, which you will see for
yourself below, so the honest model uses only `party` and `age`.

**Section bridge:** the contract is written. Now execute it, and watch how small
an honest win really is.

---

## 2. Beat the Dumbest Honest Rule on Held-Out Cases

**Guiding question:** *is my model's score any better than the baseline, and is it
scored only on cases the model never saw?*

Now you honor all four parts of the contract in order. You **split first**: a
random 25% of rows go into a held-out test set the model never sees during
training. The split uses a fixed seed so it is reproducible, and it is
**stratified**, which means both halves keep the same turnout balance. Then you
fit the **baseline** (a rule that always predicts the majority class) and a real
**model** (a logistic regression on the encoded features), and you score both on
the same held-out rows.

Because the split happened before any fitting, every number below is
**out-of-sample**: computed only on cases the model never trained on, the way a
real forecast is judged. Out-of-sample numbers are the only kind the policy
stakeholder will accept.

> **A question that often comes up here:** *"Why hold out a quarter of my data?
> Isn't that wasteful?"* The held-out rows are not wasted. They are the only
> honest test you have. Train on every row and no cases are left that the model
> did not memorize, so its score measures recall of its own homework, not skill
> on the unseen. Spending 25% of the data to earn a number a skeptic will believe
> is the opposite of waste.

### 🔮 Pause & Predict

The baseline always guesses the majority answer, "this person voted," for every
held-out voter. Before running the next cell, commit a number: what accuracy will
that baseline get on the held-out rows? Write the number and one sentence of
reasoning. You saw the whole-file base rate a moment ago, so this is not a wild
guess.

### YOUR ANSWER HERE:

**My predicted baseline accuracy (a number):**

**Why:**

---

### 🛠️ Run the Study: baseline first, then try to beat it

Run the cell below. It splits the data, scores the honest baseline on held-out
rows, then scores a logistic model on the *same* held-out rows. Watch the two
numbers and the gap between them.

**🔴 Live in class: we run this one together.**
**Before you ask:** in one sentence, predict whether the model's margin over the
baseline will be large or small, and why 2012 turnout is hard to forecast from
party and age alone.

> 💡 **Gemini Prompt:** "Here is a Python cell that holds out 25% of voter rows
> (stratified, fixed seed), scores a majority-class baseline on them, then scores
> a logistic model on the SAME held-out rows: [paste the next cell]. Explain, line
> by line, what `stratify=y` does and why the split has to happen before any
> fitting. Then give me one independent way to confirm the held-out rows were
> never in training, without rerunning your own explanation."
>
> **After running, verify (counters *confident fabrication*: a fluent walkthrough
> can quote a margin the code never produced):**
> - [ ] Read the printed baseline accuracy, model accuracy, and margin off your
>   own output, and confirm Gemini's numbers match those, not ones it guessed.
> - [ ] Run its "independent way to confirm" yourself. The self-check cell already
>   proves the train and test rows do not overlap; make sure its claim agrees.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# The reveal — run AFTER committing your prediction above.
y = voters["voted_2012"]
# CLEAN features: one-hot the party text, keep age; DROP the census-tract ID code.
X = pd.get_dummies(voters[["party", "age"]], columns=["party"], drop_first=True).astype(float)

# SPLIT: hold out 25%, fixed seed, stratified so turnout balance is preserved.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)
print(f"train rows: {len(y_train)}   held-out rows: {len(y_test)}   "
      f"held-out base rate: {y_test.mean():.1%}")

# BASELINE: always predict the majority class.
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
base_acc = accuracy_score(y_test, baseline.predict(X_test))

# MODEL: logistic regression on the standardized features.
model = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000)).fit(X_train, y_train)
model_acc = accuracy_score(y_test, model.predict(X_test))

print()
print(f"  baseline held-out accuracy: {base_acc:.3f}")
print(f"  model    held-out accuracy: {model_acc:.3f}")
print(f"  margin (model - baseline):  {model_acc - base_acc:+.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(["Baseline\n(always 'voted')", "Logistic\nmodel"],
              [base_acc, model_acc],
              color=["#8C8C8C", "#4C72B0"], edgecolor="white")
ax.axhline(base_acc, color="#8C8C8C", linestyle=":", linewidth=1.5)
ax.set_ylim(0, 1)
ax.set_ylabel("Held-out accuracy")
ax.set_title("An honest win is a small win: the model barely clears the baseline")
for b, v in zip(bars, [base_acc, model_acc]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the guarantees the contract is supposed to give us.
assert set(X_test.index).isdisjoint(set(X_train.index)), \
    "held-out rows leaked into training — the split is broken!"
assert model_acc >= base_acc, "model did not beat the baseline — report that honestly"
print(f"✓ Self-check passed: the {len(y_test)} test rows were never trained on, "
      f"so the model's {model_acc:.1%} is fully out-of-sample.")
print(f"✓ The model beats the baseline by {model_acc - base_acc:.1%} — "
      "small, and that smallness is the lesson.")

### 🔍 Reading the Evidence: write both headlines

The model scored about **0.648**; the baseline scored about **0.604**. That is a
real edge of roughly **four percentage points**, and it is *small*. A modest true
gap is worth more than a dramatic misleading one. In the cell below, write the
**honest headline** first, the one that names the baseline and the margin. Then
write the **dishonest headline** you could technically get away with, and strike
it through, so you feel exactly what you are refusing to publish.

### YOUR ANSWER HERE:

**Honest headline (names the baseline and the margin):**

**Dishonest headline, struck through so it never ships:** ~~...~~

**The margin, in one number:**

---

## 3. The Fake Win: Data Leakage

**Guiding question:** *when a score looks too good, has the model learned the
world, or quietly found the answer key?*

> *"When a model suddenly scores far better than anyone expected, I do not get
> excited. I get suspicious. Nine times out of ten it found a shortcut: a feature
> that already contains the answer. My job is to rule out the nine before we
> celebrate the one."*
> — a **skeptical peer** reviewing your pilot

This is the puzzle from the top of the lecture, made concrete. **Data leakage**
is a feature that carries information you would not actually have at the moment
you need the forecast. Its value is only settled at or after the outcome it is
supposed to predict. Example: forecasting hospital readmission using a discharge
code that is only recorded when the patient leaves. In your book's grammar,
leakage is a **data-strategy violation**: the plan for how evidence comes to exist
quietly let the answer sneak into the inputs.

The fastest way to learn the hunt is to build a leak yourself, feel how good the
score looks, then catch it. In the next cell you construct a feature called
`post_election_contact` from the very outcome you are predicting, add it to the
model, and retrain.

> **A question that often comes up here:** *"If leakage makes the score go UP, why
> is it bad? Isn't a higher score what we want?"* The score goes up on *this* data
> and collapses on *new* data, because the leaky feature will not exist, or will
> not yet be known, when you actually need a forecast. A model that scores 90% by
> reading the answer key has learned nothing it can use on a case whose answer is
> still unknown. High-and-fake is worse than modest-and-real.

Leakage is easiest to see on a timeline. The diagram below lays the pieces left to
right in the order they happen in time. An honest feature is known *before* the
prediction moment, so it can feed the forecast. The outcome comes later. A leaky
feature is settled at or after the outcome, so an arrow from it back to the
prediction moment would carry information from the future, which no real forecast
can have.

In [ ]:
# A leakage-timeline diagram: honest inputs flow forward; a leak flows backward.
# (networkx is imported once in the setup cell.)
G = nx.DiGraph()
pos = {
    "Honest features\n(party, age)\nknown BEFORE": (0.0, 0.0),
    "Prediction\nmoment":                           (1.3, 0.0),
    "Outcome\n(did they vote?)":                    (2.6, 0.0),
    "Leaky feature\nsettled AT/AFTER\nthe outcome": (3.9, 0.0),
}
G.add_nodes_from(pos)
allowed = [("Honest features\n(party, age)\nknown BEFORE", "Prediction\nmoment"),
           ("Prediction\nmoment", "Outcome\n(did they vote?)"),
           ("Outcome\n(did they vote?)", "Leaky feature\nsettled AT/AFTER\nthe outcome")]

fig, ax = plt.subplots(figsize=(11, 4))
nx.draw_networkx_nodes(G, pos, node_color="#DCE6F1", edgecolors="#4C72B0",
                       linewidths=1.5, node_size=6800, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=allowed, arrowstyle="-|>", arrowsize=18,
                       edge_color="#555555", width=1.6, node_size=6800, ax=ax)
# The forbidden flow: from the leaky feature BACK to the prediction moment.
ax.annotate("", xy=pos["Prediction\nmoment"], xytext=pos["Leaky feature\nsettled AT/AFTER\nthe outcome"],
            arrowprops=dict(arrowstyle="-|>", color="#D1651A", lw=2.2,
                            linestyle="dashed", connectionstyle="arc3,rad=-0.45"))
ax.text(2.6, 1.15, "LEAKAGE: information from the future (not allowed)",
        color="#D1651A", ha="center", fontsize=9, fontweight="bold")
ax.set_title("Leakage on a timeline: an honest input is known before the forecast; a leak is not")
ax.set_xlim(-0.7, 4.7)
ax.set_ylim(-0.8, 1.5)
ax.axis("off")
plt.tight_layout()
plt.show()

print("✓ Leakage timeline drawn.")
print("  Solid arrows run forward in time and are allowed.")
print("  The dashed arrow runs backward from the outcome's future into the forecast.")
print("  That backward arrow is the leak, and the timing is what makes it illegal.")

*In plain terms, the diagram says a feature can only be an input if its value is
already settled at the prediction moment. The dashed arrow is the leak: it reaches
back from after the outcome, so a real forecast could never use it.*

### 🔮 Pause & Predict

Imagine a canvasser's log flag, `post_election_contact`, tallied *after* the
election from the same rolls that record turnout, so it is tangled up with whether
the person voted. The honest model scored about 0.648 held-out. Before running the
next cell, commit a number: what will held-out accuracy be once this leaky feature
joins the model? Roughly the same, a little higher, or a lot higher?

### YOUR ANSWER HERE:

**My predicted held-out accuracy with the leaky feature:**

**Why:**

---

### 🛠️ Hands-On: build the leak, watch it, then hunt it

Run the two cells below. The first builds the leaky feature from the outcome and
scores it. The second hunts it (correlation plus timing), retrains clean, and
reports the honest gap. Read the numbers before the next markdown cell.

**🏠 Homework depth: run the Gemini prompt on your own after class.**
**Before you ask:** in one sentence, name the two tests you would run on any
feature behind a suspicious score jump. Commit your own answer before the tool
offers one.

> 💡 **Gemini Prompt:** "This code builds a feature `post_election_contact` FROM
> the outcome `voted_2012` (with 15% of values flipped as noise), adds it to a
> logistic model, and retrains; a later cell drops it and rescores: [paste both
> cells]. Explain why a feature built from the outcome inflates held-out accuracy,
> then argue whether this feature could ever legitimately be used in a real
> forecast, and if not, exactly why not."
>
> **After running, verify (counters *plausible-but-wrong-method*: a fluent case
> for keeping a leaky feature is still the wrong method for a real forecast):**
> - [ ] Check the printed leaky accuracy and the leaky-minus-clean gap against
>   Gemini's account. Does dropping the leak collapse the score back to about
>   0.648, as it should?
> - [ ] Reject any suggestion that a feature settled *after* the outcome can stay
>   in the model. The timing test, not the accuracy, decides. Re-derive the timing
>   yourself.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# You build a leak ON PURPOSE so you can practice catching one.
# `post_election_contact` is the outcome with 15% of its values flipped as noise.
# In a real project no feature arrives labeled 'leak'; the tell is that its value
# depends on the outcome. A FRESH seeded generator makes this reproducible.
leak_rng = np.random.default_rng(SEED)
flip = leak_rng.random(len(voters)) < 0.15
voters["post_election_contact"] = np.where(flip, 1 - voters["voted_2012"],
                                           voters["voted_2012"]).astype(float)

X_leaky = X.assign(post_election_contact=voters["post_election_contact"].values)
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky, y, test_size=0.25, random_state=SEED, stratify=y)

leaky_model = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=1000)).fit(Xl_train, yl_train)
leaky_acc = accuracy_score(yl_test, leaky_model.predict(Xl_test))
print(f"  model WITH the leaky feature — held-out accuracy: {leaky_acc:.3f}")
print("  (that jump should make you suspicious, not proud)")

In [ ]:
# The score jump was the ALARM. Now the two checks that confirm a leak:
# CHECK 1 (correlation) — how tightly does the suspect feature track the outcome?
r = np.corrcoef(voters["post_election_contact"], voters["voted_2012"])[0, 1]
print(f"corr(post_election_contact, voted_2012) = {r:.3f}   "
      "-> almost a copy of the answer")
# CHECK 2 (timing) — it is tallied AFTER the election, so no real forecast could use it.
print("timing: recorded AFTER turnout is known -> not available at prediction time")
print()

# RETRAIN CLEAN — drop the leak, score honestly again (this is the model from section 2).
clean_acc = model_acc
print(f"  leaky held-out accuracy: {leaky_acc:.3f}")
print(f"  clean held-out accuracy: {clean_acc:.3f}")
print(f"  inflation the leak bought: {leaky_acc - clean_acc:+.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(["Clean model\n(honest)", "Leaky model\n(reads the answer)"],
              [clean_acc, leaky_acc],
              color=["#4C72B0", "#D1651A"], edgecolor="white",
              hatch=["", "//"])
ax.axhline(base_acc, color="#8C8C8C", linestyle=":", linewidth=1.5,
           label=f"baseline = {base_acc:.3f}")
ax.set_ylim(0, 1)
ax.set_ylabel("Held-out accuracy")
ax.set_title("The leak fakes a huge win; the hatched bar is not real skill")
ax.legend(loc="lower right")
for b, v in zip(bars, [clean_acc, leaky_acc]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the leak really did inflate the score, and both numbers are honest.
assert leaky_acc > clean_acc + 0.15, "the leak should inflate the score sharply"
assert set(Xl_test.index).isdisjoint(set(Xl_train.index)), "held-out rows leaked into training!"
print(f"✓ Self-check passed: the leak inflated held-out accuracy by "
      f"{leaky_acc - clean_acc:.1%}, from {clean_acc:.1%} to {leaky_acc:.1%}.")
print("✓ Pure illusion: the feature encodes the answer, so the jump is not skill.")

### 🔍 Reading the Evidence: prediction is not explanation

The leak jumped held-out accuracy from about 0.648 to about **0.892**, roughly 24
points, by smuggling the answer into the inputs. If you ranked the leaky model's
features by weight, `post_election_contact` would tower over party and age and
look like "the most important driver of voting." It is nothing of the kind. It is
the answer, copied.

This is the line the whole week defends. **Interpretability** is reading which
features a model leaned on. **Explanation** is knowing what actually causes the
outcome. They are not the same, and a leak fakes the first while telling you
nothing about the second. In the cell below, write two things. First: what did the
leaky model's top feature teach you about *why* people vote? Second: write one
sentence the clean model's result *permits*, and one it does *not* permit no
matter how its weights rank.

### YOUR ANSWER HERE:

**What the leaky model's top feature taught me about why people vote:**

**A claim the clean model permits:**

**A claim it does NOT permit (an explanation read off the weights):**

---

---

### ⏸ In class through here

**You have reached the end of the in-class block.** Everything below deepens the
same ideas and feeds your M7 declared analysis protocol. Finish it before Friday's
studio: the checks that separate a real win from a lucky one (cross-validation,
overfitting, calibration, subgroup, distribution shift), the metric design choice,
the practice drill, the AI interrogation and human-only verdict, and your
project's prediction spine. Come back with your target, baseline, and metric
chosen, or a defended reason prediction is the wrong question.

---

## 4. The Checks That Separate a Real Win From a Lucky One

**Guiding question:** *my model beat the baseline by four points on one split, so
how do I know that gain is real, stable, and fair, rather than luck?*

A single held-out number is a start, not a verdict. Five checks turn "it beat the
baseline once" into "I would defend this forecast." You will run each one on the
clean model. They are the backbone of your M7 declared analysis protocol, so name
them as you go: **overfitting**, **cross-validation**, **calibration**,
**subgroup performance**, and **distribution shift**.

**Overfitting** is when a model memorizes the training rows instead of learning a
pattern that carries to new cases. The tell is a large gap between a high
training score and a low held-out score. Example: give the model hundreds of
census-tract ID columns and it can practically look up each training voter, yet it
learns nothing that transfers. Watch it happen.

### 🔮 Pause & Predict

The next cell adds every census tract as its own 0/1 column, hundreds of them, and
fits the same kind of model. Before running it, commit two numbers: the model's
*training* accuracy and its *held-out* accuracy. Will the two numbers stay close,
the way the clean model's roughly did, or split far apart, and in which direction?
Write both numbers and one sentence of reasoning.

### YOUR ANSWER HERE:

**My predicted training accuracy and held-out accuracy for the tract-heavy model:**

**Why:**

---

### 🛠️ Hands-On: watch a model memorize

Run the cell. It fits the honest clean model and the tract-heavy model, and plots
each one's training accuracy next to its held-out accuracy.

**🏠 Homework depth: run the Gemini prompt on your own.**
**Before you ask:** commit your own one-sentence answer first. Overfitting is the
name for this gap, but what else could make a model do great on training rows and
poorly on new ones?

> 💡 **Gemini Prompt:** "This cell fits two models and prints each one's training
> accuracy and held-out accuracy: a clean model on party and age (13 columns), and
> a tract-heavy model with hundreds of census-tract ID columns: [paste the next
> cell and its four printed numbers]. Overfitting is my leading explanation for the
> high-training, low-held-out gap. List every OTHER explanation that could produce
> that same gap here, such as a broken split, a leaked feature, a coding bug, or a
> lopsided class balance, and for each name the single check I could run to rule it
> in or out."
>
> **After running, verify (counters *illusion of completeness*: a fluent list can
> read as the full set of causes while quietly skipping the check that would settle
> it):**
> - [ ] Read the four printed numbers (each model's train and held-out accuracy) and
>   confirm every cause Gemini lists is consistent with those exact values, not ones
>   it invented.
> - [ ] For the alternative you find most plausible, run its check yourself (for the
>   split, rerun the disjoint-index self-check below) and confirm overfitting, not
>   that alternative, is what your numbers show.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Overfitting: give the model hundreds of ID columns and watch it memorize.
X_tract = pd.get_dummies(voters[["party", "age", "census_tract"]],
                         columns=["party", "census_tract"], drop_first=True).astype(float)
print(f"tract-heavy feature count: {X_tract.shape[1]} columns "
      f"(vs {X.shape[1]} for the clean model)")

Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    X_tract, y, test_size=0.25, random_state=SEED, stratify=y)
overfit_model = make_pipeline(StandardScaler(),
                              LogisticRegression(max_iter=2000)).fit(Xt_train, yt_train)

clean_train = accuracy_score(y_train, model.predict(X_train))
of_train = accuracy_score(yt_train, overfit_model.predict(Xt_train))
of_test = accuracy_score(yt_test, overfit_model.predict(Xt_test))
print(f"  clean model    — training: {clean_train:.3f}   held-out: {model_acc:.3f}")
print(f"  tract-heavy    — training: {of_train:.3f}   held-out: {of_test:.3f}")

xpos = np.arange(2)
w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(xpos - w / 2, [clean_train, of_train], w, label="training accuracy",
       color="#A6A6A6", edgecolor="white")
ax.bar(xpos + w / 2, [model_acc, of_test], w, label="held-out accuracy",
       color="#4C72B0", edgecolor="white")
ax.axhline(base_acc, color="#8C8C8C", linestyle=":", linewidth=1.5,
           label=f"baseline = {base_acc:.3f}")
ax.set_xticks(xpos)
ax.set_xticklabels(["Clean model\n(party + age)", "Tract-heavy model\n(hundreds of ID columns)"])
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_title("Overfitting: memorizing training rows does not carry to held-out cases")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the tract-heavy model overfits — huge train/held-out gap, and it
# actually does WORSE than the dumb baseline on held-out rows.
assert of_train - of_test > 0.25, "expected a large train/held-out gap"
assert of_test < base_acc, "the overfit model should fall below the baseline held-out"
print(f"✓ Self-check passed: the tract-heavy model trains at {of_train:.1%} but scores "
      f"only {of_test:.1%} held-out,\n  below the {base_acc:.1%} baseline. That gap is overfitting.")

The tract-heavy model trained at about **0.94** and scored only about **0.58**
held-out, below the 0.604 baseline. It memorized the training voters and learned
nothing that transfers. The clean model's honest **0.648** is worth more than that
fake 0.94. The held-out test is what exposed the fraud, which is exactly why the
split came first.

One held-out split can still be a lucky draw. **Cross-validation** guards against
that. It splits the data into several equal parts (folds), trains on all but one,
scores on the held-out fold, and rotates so every row is tested once. Example:
five folds give five held-out scores, and their average is a sturdier estimate
than any single split. Run it on the clean model.

In [ ]:
# Cross-validation: five held-out scores instead of one lucky split.
cv_scores = cross_val_score(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    X, y, cv=5, scoring="accuracy")
print("Five-fold held-out accuracies:", np.round(cv_scores, 3).tolist())
print(f"  mean = {cv_scores.mean():.3f}   spread (std) = {cv_scores.std():.3f}")
print(f"  baseline to beat: {base_acc:.3f}")

The five folds land around **0.613** on average with a small spread of about
**0.015**, all just above the baseline. In plain terms, the modest win is real and
stable, not an artifact of one lucky split. It is also honestly small, and a
cross-validated small win is still a win worth reporting with its uncertainty.

Accuracy is not the only thing that matters. **Calibration** asks whether the
model's stated probabilities mean what they say. A model is well calibrated when,
among the cases it calls 70% likely, about 70% actually happen. Example: a weather
app that says 70% rain and is right 70% of those days is calibrated; one that says
90% and is right half the time is not. Check the clean model.

In [ ]:
# Calibration: do the model's probabilities mean what they say?
proba = model.predict_proba(X_test)[:, 1]
frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=5, strategy="quantile")
cal_table = pd.DataFrame({"mean_predicted_prob": np.round(mean_pred, 3),
                          "observed_share_voted": np.round(frac_pos, 3)})
print("Grouped from least to most confident (about 50 held-out voters per row):")
print(cal_table.to_string(index=False))
print(f"\nThe model's predicted probabilities span only "
      f"{proba.min():.2f} to {proba.max():.2f}.")

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], color="#8C8C8C", linestyle="--", label="perfect calibration")
ax.plot(mean_pred, frac_pos, marker="o", color="#4C72B0", label="this model")
ax.set_xlabel("Mean predicted probability of voting")
ax.set_ylabel("Observed share who actually voted")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Calibration (reliability) check on held-out voters")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Reading the calibration.** The dots roughly follow the diagonal in the middle,
so the model is only roughly calibrated, and it is cautious. Its probabilities
never leave the narrow band from about 0.25 to 0.99, and most sit near 0.5 to 0.7,
so it rarely commits to a confident call. Each bin holds only about 50 voters, so
part of the wobble is noise, not miscalibration. The honest takeaway: this model
is fine for ranking who is more likely to vote, but you should not read its
"0.63" as a precise probability for any one person.

> **A question that often comes up here:** *"My model is accurate. Why do I care
> whether its probabilities are calibrated?"* Because the *decision* you build on
> top of the score depends on it. If you send help to everyone the model calls
> "over 80% at risk," a model that never actually reaches 80%, or that says 80%
> when the truth is 50%, will send help to the wrong people. Accuracy grades the
> label; calibration grades the number you act on.

A single overall score can also hide who the model fails. **Subgroup
performance** means checking the metric separately within groups that matter, not
just on everyone at once. Example: a model can look fine overall while quietly
being much worse for one party or one age band. Check accuracy by party.

In [ ]:
# Subgroup performance: does the overall accuracy hide a group it fails?
holdout = voters.loc[X_test.index].copy()
holdout["predicted"] = model.predict(X_test)
holdout["truth"] = y_test.values
holdout["correct"] = holdout["predicted"] == holdout["truth"]
subgroup = (holdout.groupby("party")
            .agg(n=("correct", "size"),
                 accuracy=("correct", "mean"),
                 true_turnout=("truth", "mean")))
subgroup = subgroup[subgroup["n"] >= 20].sort_values("accuracy")
print("Held-out accuracy by party (groups with at least 20 held-out voters):")
print(subgroup.round(3).to_string())
print(f"\nOverall held-out accuracy for comparison: {model_acc:.3f}")

**Reading the subgroups.** Accuracy ranges from about 0.62 to 0.74 across
parties, so the single 0.648 headline smooths over real differences. The sharpest
flag is the no-party-preference (NPP) group: its true turnout is about 0.38, far
below the overall 0.60, so a model tuned to the average base rate misreads that
group the most. If your project's decision affects a subgroup, its subgroup score,
not the overall one, is the number you must defend.

The last check reaches past your data entirely. **Distribution shift** is when the
cases you deploy on differ from the cases you trained on, so a fair-looking model
degrades in the real world. Example: train on younger voters, deploy on older
ones. Watch the accuracy fall.

In [ ]:
# Distribution shift: train on younger voters, deploy on older ones.
median_age = voters["age"].median()
younger = voters["age"] < median_age
X_young, y_young = X.loc[younger], y.loc[younger]
X_old, y_old = X.loc[~younger], y.loc[~younger]

# Train ONE model on younger voters (held out 25% of them for an honest in-group test).
Xy_tr, Xy_te, yy_tr, yy_te = train_test_split(
    X_young, y_young, test_size=0.25, random_state=SEED, stratify=y_young)
young_model = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=1000)).fit(Xy_tr, yy_tr)

in_dist_acc = accuracy_score(yy_te, young_model.predict(Xy_te))     # same population
shifted_acc = accuracy_score(y_old, young_model.predict(X_old))     # shifted population
print(f"trained on voters younger than {median_age:.0f}")
print(f"  in-distribution accuracy (held-out YOUNGER voters): {in_dist_acc:.3f}")
print(f"  shifted accuracy (OLDER voters it never saw):       {shifted_acc:.3f}")
print(f"  older-voter base rate for comparison:               {y_old.mean():.3f}")

**Reading the shift.** On younger voters like the ones it trained on, the model
scored about **0.685**. Pointed at older voters, it fell to about **0.591**, which
is essentially the older group's base rate. The model did not break; the world
moved. A forecast is only licensed for cases that resemble its training data, and
naming that boundary is part of an honest prediction.

> **A question that often comes up here:** *"If my held-out score is good, am I not
> safe to deploy?"* Only if tomorrow's cases resemble today's. A held-out set drawn
> from the same pool cannot warn you about a shift, because it shares the pool's
> quirks. The check is to ask, in words, how the deployment cases might differ from
> the training cases, and to watch the score on the groups you actually expect to
> face.

### ⚖️ Make a Design Choice: pick the metric that matches the job

*(Human-first: settle your own choice and its defense before you ask any AI.)*

Accuracy hides *which* class a model gets right. The cell below prints a
**confusion matrix**, a small table of truth versus guess (rows are what really
happened, columns are what the model predicted), plus each class's **recall** (of
the true cases of a class, what share did the model catch) and **precision** (of
the cases it flagged as a class, what share truly were).

**🏠 Homework depth: run the Gemini prompt on your own.**
**Before you ask:** in one sentence, name which error costs a get-out-the-vote
campaign more, calling a real non-voter a voter, or the reverse.

> 💡 **Gemini Prompt:** "I am running a get-out-the-vote campaign and want to find
> the NON-voters so I can reach them. Here is my confusion matrix and per-class
> precision and recall on held-out voters: [paste the printed output]. List every
> metric that could score this task and, for each, say in one line what it would
> reward and what it would ignore for my goal of finding non-voters."
>
> **After running, verify (counters *illusion of completeness*: a tidy metric list
> can quietly omit the one metric your goal actually needs):**
> - [ ] Check the list against your goal yourself: since you want to *find
>   non-voters*, recall on the non-voter class is the one that matters. If the list
>   buries or omits it, the tidy output failed the one test it looked complete on.
> - [ ] Confirm each metric it names matches a number in your printed report, and
>   discard any it invented that is not in your output.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Where do the model's right and wrong answers fall?
cm = confusion_matrix(y_test, model.predict(X_test))
print("Confusion matrix (rows = truth, cols = prediction):")
print(pd.DataFrame(cm,
      index=["truth: did NOT vote", "truth: voted"],
      columns=["pred: did NOT vote", "pred: voted"]))
print()
print(classification_report(y_test, model.predict(X_test),
      target_names=["did not vote (0)", "voted (1)"]))

The model catches most actual voters but only a small share of actual
non-voters. It leans toward the popular answer. Now suppose you run a
get-out-the-vote campaign and want to **find the non-voters** so you can reach
them. Accuracy would reward the model for nailing the voters you were *not* trying
to find. In the cell below, choose the metric you would optimize for the campaign,
(a) overall accuracy, (b) recall on the non-voter class, or (c) precision on the
non-voter class, and defend the choice in one paragraph tied to what a wrong
answer *costs* the campaign.

### YOUR ANSWER HERE:

**Metric I would optimize (a / b / c):**

**One-paragraph defense (what a false positive vs a false negative costs here):**

---

### 📝 Practice: match the metric, then rank the leaks

*(Human-first: do both parts yourself before any AI. The sorting is the graded
skill.)*

**Part 1, match the metric to the target.** For each, name one metric you would
trust and one you would distrust, in a sentence. The shape of the target picks the
scorecard.

- **A.** Flagging fraudulent transactions, where about 1 in 1,000 is fraud.
- **B.** Predicting the winner of a near 50/50 A/B test outcome.
- **C.** Flagging patients at risk of a dangerous complication, where a missed case
  can be fatal but a false alarm just triggers a second check.

**Part 2, rank four features by leakage risk.** You predict, at the moment a
patient is **admitted**, whether they will be **readmitted within 30 days of
discharge**. Rank these from highest to lowest leakage risk, and give the one-word
reason (timing) for the top and bottom.

- **D.** Patient's age at admission.
- **E.** Number of prior admissions in the previous year.
- **F.** Discharge disposition code, recorded *at discharge*.
- **G.** Total medications administered during the stay, finalized *at discharge*.

### YOUR ANSWER HERE:

**Part 1, A (rare event):**

**Part 1, B (balanced):**

**Part 1, C (costly misses):**

**Part 2, ranking (highest to lowest leakage risk):**

**Part 2, reason for the highest and the lowest:**

---

## 5. The Verdict: Is This Model Useful?

**Guiding question:** *with all the checks in hand, how do you decide to keep a
model, and when is the honest call to reject it?*

The five checks give you a **model-selection rule**: a written standard for keeping
or rejecting a model, fixed before you see which one you happen to like. Example:
"keep a model only if it beats the baseline under cross-validation and survives a
leakage check." Choosing the rule before you peek is what stops you from crowning
whichever model flatters your story.

Those same checks tell you when a model is **not useful**, a verdict worth as much
as a success. Declare a model not useful when any one of these holds:

- It does not beat the baseline out-of-sample. The base rate was already free.
- The win vanishes under cross-validation, so it rode on one lucky split.
- The win depends on a leaked feature that would not exist at prediction time.
- It is badly calibrated, so the probability you would act on is misleading.
- It fails on the subgroup or the shifted population you actually care about.

"Prediction is the wrong tool for this question" is an honest, gradable finding,
not a failure. Your M7 protocol is judged on the reasoning behind the verdict, not
on whether the verdict is yes.

> **A question that often comes up here:** *"If I decide my rule after seeing all
> the scores, is that not just being thorough?"* No, it is how you fool yourself.
> Picking the metric and the model after you have seen which combination wins lets
> you rationalize the choice that looks best on this data. Declaring the rule first,
> the way M7 asks, is what makes the win believable to a skeptic.

### 🔁 Modify the Prompt

Earlier you had Gemini reason about a leaky feature the notebook handed you. Now
point the same move at *your own* project, where the timeline is yours to know and
the tool's is only a guess. First, in one sentence, name the single feature in your
data you most suspect could leak. Do not let AI pick it. Then adapt the base prompt
so it checks *your* feature list against *your* outcome's timing.

> **Base prompt (the leakage-timing check):** "Here is the outcome I want to predict
> and the moment I would need the forecast: [your outcome, and when it is decided].
> Here is my candidate feature list with when each value is recorded: [your list].
> For each feature, tell me whether its value is settled before, at, or after the
> outcome, and flag any that could not exist at prediction time."

In the cell below: paste your adapted prompt, predict which feature Gemini will flag
first, then run it and note whether its flag matched the one you named. If it clears
a feature you know is settled late, that is the tool failing the timing test, and
you keep your own judgment.

*(Homework depth: run this against your real project before Friday's studio.)*

### YOUR ANSWER HERE:

**The feature I most suspect could leak (I named it before asking):**

**My adapted leakage-timing prompt (my outcome, my feature list, my timeline):**

**Which feature I predicted Gemini would flag first:**

**What Gemini actually flagged, and whether I trust it against my own timeline:**

---

### 🔬 Interrogate the Output

Suppose Gemini returns a confident verdict on your model, "this is a strong,
deployable predictor." Do not accept it, and do not reject it on reflex either.
Interrogate the verdict against your own numbers, using four checks. Answer each in
the cell below.

- **Claims:** does the verdict match your printout? Your clean model beat the
  baseline by about four points and dropped below it once overfit, so a claim of
  "strong predictor" has to reckon with a four-point edge, not a triumph.
- **Assumptions:** what does the verdict assume about the deployment cases? If it
  ignores the distribution-shift result (about 0.685 on younger, about 0.591 on
  older), it is assuming a world that does not shift.
- **Missing information:** which of the five checks did the verdict skip? A verdict
  that never mentions leakage, calibration, or the subgroup gap left out the checks
  that decide usefulness.
- **Overstatements:** does any word claim more than a four-point cross-validated
  edge supports? Flag the exact phrase.

And the rule that governs the whole notebook: **code that runs without an error is
not the same as code that computed the right thing.** A cell can execute cleanly and
still score a leaked feature. Verify the *number* and its *meaning*, not the green
check.

*(Homework depth.)*

### YOUR ANSWER HERE:

**Claims (does the verdict match my numbers?):**

**Assumptions (did it ignore the shift, or a subgroup?):**

**Missing information (which of the five checks did it skip?):**

**Overstatements (exact words that claim too much):**

---

### 🧑‍⚖️ Human-Only Checkpoint

Close Gemini for this one. No AI, no lookup. This is a never-delegate decision: the
verdict on whether your project should predict at all, and if so, on what terms.
That call is yours to declare and defend. In the cell below, write, in your own
words:

1. **Does your project have a real prediction question?** Yes means there is a
   future or unseen case whose outcome you would forecast. No means your real
   question is description, generalization, or a cause.
2. **If yes:** name your **target** (the exact column), the **baseline** you must
   beat, the **metric** that fits the target's shape, and the one feature most
   likely to **leak**, with the timing check you will run to rule it out.
3. **If no:** write one sentence on why prediction is the wrong column for your
   question, then name the too-good-to-be-true trap nearest your actual approach, a
   measure that quietly contains its own outcome, and your check for it.

Both verdicts are correct if they are defended. If you cannot yet name a baseline
you would have to beat, that is a finding: it tells you the prediction question is
not yet real.

### YOUR ANSWER HERE:

**Does my project have a real prediction question? (yes / no):**

**If yes (target / baseline / metric):**

**The feature most likely to leak + my timing check:**

**If no (why prediction is the wrong column + the nearest look-alike trap + my check):**

---

### 🎯 Project Transfer: the spine of your M7 declared analysis protocol

*(Human-first: draft every piece yourself. AI may critique it after, but the
protocol you defend at the studio has to be reasoned by you.)*

This is where the week becomes your **M7 declared analysis protocol**: a written
statement of exactly how you will analyze your data, fixed *before* you touch it, so
no result can be reverse-engineered into a flattering method. In the cell below,
draft the protocol's spine for *your* project:

1. **The prediction verdict:** does your project predict? If yes, state the target,
   baseline, split plan, and metric, in that order. If no, state the question it
   really asks and the design that answers it.
2. **The nearest leakage risk:** the one feature (or measure) whose value could be
   settled at or after your outcome, and the timing check that rules it out.
3. **The honesty checks you will run before reporting anything:** which of the five
   (cross-validation, overfitting, calibration, subgroup, distribution shift) apply
   to your design, and what result on each would make you retract the claim.
4. **The claim boundary (one sentence):** the exact forecast or finding your
   analysis would license, worded to stop where your evidence stops, with its
   uncertainty.

These four pieces are what the prediction-and-leakage auditor will attack at
Friday's studio. Write them so a skeptic who has never seen your project could find
the exact edge of what it can claim.

### YOUR ANSWER HERE:

**1. The prediction verdict (target / baseline / split / metric, or the honest no):**

**2. The nearest leakage risk + my timing check:**

**3. The honesty checks that apply + the result that would make me retract:**

**4. My claim boundary, in one sentence (with its uncertainty):**

---

### 📒 AI Research Ledger

Log every AI use from this notebook in the ledger. One worked row is filled in as a
model, and it captures the discipline this week teaches: **you set the goal, the AI
listed options, and you checked the list against your own printout.** This notebook
offers four prompts, one live in class and three at homework depth, so your ledger
carries a row for each one you actually ran, not a fixed count. The ledger is a
graded habit, not paperwork. It is how you show your work was verified.

| Task delegated | Tool used | Prompt | Output summary | Decision | Verification method | Remaining concern | Responsible researcher |
|---|---|---|---|---|---|---|---|
| List every metric that could score the get-out-the-vote task | Gemini | "I want to find the non-voters. Here is my confusion matrix and per-class precision and recall: [pasted]. List every metric and what each rewards." | A tidy, confident list of six metrics that buried recall on the non-voter class near the bottom | Chose recall on the non-voter class myself, because my goal is to *catch* non-voters | Checked each metric against my printed report; matched my goal (find non-voters) to recall, not accuracy | The list looked complete, so I nearly accepted its default ordering | *(your name)* |
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |

---

### 🛡️ Exit Defense

Defense #07 — write, in your own words:

1. **The claim I can defend:** one bounded sentence, from today's data or your own
   project, that you would put your name on (a forecast with its baseline and
   margin, or an honest "prediction is the wrong tool because ...").
2. **Its boundary:** what this evidence does NOT establish. Name the compass
   position (prediction, descriptive kind, unseen-cases reach) and confirm you make
   no causal reading of any weight.
3. **My uncertainty and limitations:** one sentence naming your leakage risk, the
   spread across cross-validation folds, or a subgroup or shift you have not tested.
4. **AI check:** what you delegated to Gemini, whether you **accepted, changed, or
   rejected** what it returned, and the specific check that decided it (a margin you
   recomputed, a metric you matched to your goal, a feature's timing you re-derived).

---

### YOUR ANSWER HERE:

**1. The claim I can defend:**

**2. Its boundary (compass position + no causal reading):**

**3. My uncertainty and limitations:**

**4. AI check (what I delegated, how I verified):**

---

## 6. Wrap-Up

In one lecture you ran prediction the honest way. You signed the four-part contract,
target then baseline then split then metric, and watched a logistic model beat the
majority-class baseline by only about four points on held-out voters. That taught
you the week's first lesson: an honest small win beats a dramatic misleading one.
Then you built a leak on purpose, felt the fake climb to about 0.89, hunted it by
correlation and timing, and retrained clean. You stress-tested the honest win with
five checks, cross-validation, overfitting, calibration, subgroups, and
distribution shift, and turned "it beat the baseline once" into a verdict you could
defend or honestly reject. And you drew the hard line: a model's weights are not
mechanisms, so prediction answers *who*, never *why*.

Notice what you never needed: a fifth framework. Prediction filled the book's own
declare-diagnose-redesign grammar from the start, an inquiry about unseen cases, a
data strategy that forbids leakage, and a held-out diagnosis. That is exactly why
this course could author the missing library entry, "Observational: predictive," in
the book's own voice rather than bolt a separate unit onto the side of the compass.

> **"A prediction score means nothing until you name the baseline it beat and prove
> the model never saw the answer. High-and-fake is worse than modest-and-real."**

Next week the harder word arrives. Prediction was content to forecast *who*; causal
reasoning insists on *because*, and earning it takes random assignment and an
identification argument you can defend. Bring your M7 protocol to Friday's studio,
where the prediction-and-leakage auditor will test it, and then carry your prediction
verdict into the experimental-causal pathway. This notebook companions the
course-authored "Observational: predictive" library entry and sits alongside the
book's Part III design library and its ch. 2 MIDA framing.

---

## 7. Sources & Provenance

**Provenance of this notebook:**
- *Course-authored 'Observational: predictive' library entry (declare-diagnose-redesign format, modeled on the DeclareDesign design library at book.declaredesign.org/library) | prediction as the design-library entry the book stops short of | authored in the book's grammar | fresh (course-authored)*
- *scikit-learn | the train/held-out split, majority-class baseline, logistic model, metrics, cross-validation, calibration, and the leakage/overfitting/shift demonstrations | built for the course | fresh*
- *la_voter_file.csv | rdss package data | 2012 turnout predicted from party and age; a leaky feature built from the outcome; overfitting shown by adding hundreds of tract-ID columns | adapted (used here as a prediction target; the same file is nb04's sampling frame)*
- *fresh | the four-part contract, the five-check verdict framework, and the criteria for declaring a model not useful | newly-constructed-from-source-concept*
- *callingbullshit.org (public index) | big-data and algorithm-hype material | linked as optional study, index pages only | referenced*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, Part III (the design library) and ch. 2 (the MIDA framework). Free
  online: [book.declaredesign.org](https://book.declaredesign.org/). Part III stops
  before prediction, so the "Observational: predictive" entry is course-authored in
  the book's declare-diagnose-redesign grammar; the mechanics are built fresh with
  scikit-learn.
- *(Optional)* Bergstrom, C. & West, J. *Calling Bullshit*, the big-data and
  algorithm module: [callingbullshit.org/tools](https://callingbullshit.org/tools.html).

---

<center>

Thank you!

</center>